In [1]:
from sympy import *
from IPython.display import display
import numpy as np

M = 1   # Floquet truncation order

t, theta, omega_p = symbols('t theta omega_p', real=True)
k, n              = symbols('k n', integer=True)
omega             = IndexedBase('omega')   # omega[q] = frequency of mode q

Zs0 = symbols('Zs0', complex=True)
Yg0 = symbols('Yg0', complex=True)

# Zs^(m) and Yg^(m) as Functions of a frequency argument
# e.g. Zs_m[0](omega[k-1])  displays as  Zs1(omega[k-1])
Zs_m = [Function(f'Zs{m}') for m in range(1, M+1)]
Yg_m = [Function(f'Yg{m}') for m in range(1, M+1)]

V  = IndexedBase('V')
Ic = IndexedBase('I')

# Shorthand for the Fourier basis element
def E(j):
    return exp(I * j * omega_p * t)

xi = symbols('xi')   # placeholder frequency argument

def Zs_series(t):
    """Zs(t) with xi as placeholder frequency argument."""
    s = Zs0
    for mi, Zm in enumerate(Zs_m, start=1):
        s += Zm(xi) * E(mi) * exp( I*mi*theta) \
           + Zm(xi) * E(-mi) * exp(-I*mi*theta)
    return s

def Yg_series(t):
    """Yg(t) with xi as placeholder frequency argument."""
    s = Yg0
    for mi, Ym in enumerate(Yg_m, start=1):
        s += Ym(xi) * E(mi) * exp( I*mi*theta) \
           + Ym(xi) * E(-mi) * exp(-I*mi*theta)
    return s

Zs_t = Zs_series(t)
Yg_t = Yg_series(t)

Vn1_t = V[k, n]  - Zs_t * Ic[k, n]       # Zs(t) is the only time-varying thing
In1_t = Ic[k, n] - Yg_t * Vn1_t

# Expand so every term is a monomial in E(j) = exp(I*j*omega_p*t)
Vn1_expanded = expand(Vn1_t)
In1_expanded = expand(In1_t)

display(Vn1_expanded)
display(In1_expanded)


-Zs0*I[k, n] - Zs1(xi)*exp(I*theta)*exp(I*omega_p*t)*I[k, n] - Zs1(xi)*exp(-I*theta)*exp(-I*omega_p*t)*I[k, n] + V[k, n]

Yg0*Zs0*I[k, n] + Yg0*Zs1(xi)*exp(I*theta)*exp(I*omega_p*t)*I[k, n] + Yg0*Zs1(xi)*exp(-I*theta)*exp(-I*omega_p*t)*I[k, n] - Yg0*V[k, n] + Zs0*Yg1(xi)*exp(I*theta)*exp(I*omega_p*t)*I[k, n] + Zs0*Yg1(xi)*exp(-I*theta)*exp(-I*omega_p*t)*I[k, n] + Yg1(xi)*Zs1(xi)*exp(2*I*theta)*exp(2*I*omega_p*t)*I[k, n] + 2*Yg1(xi)*Zs1(xi)*I[k, n] + Yg1(xi)*Zs1(xi)*exp(-2*I*theta)*exp(-2*I*omega_p*t)*I[k, n] - Yg1(xi)*exp(I*theta)*exp(I*omega_p*t)*V[k, n] - Yg1(xi)*exp(-I*theta)*exp(-I*omega_p*t)*V[k, n] + I[k, n]

In [6]:
js_nonzero = [j for j in range(-2*M, 2*M+1) if j != 0]
results = {}

# non-zero harmonics
for j_val in js_nonzero:
    cv_raw = Vn1_expanded.coeff(E(j_val))
    ci_raw = In1_expanded.coeff(E(j_val))
    cv = cv_raw.subs(xi, omega[k]).subs(k, k - j_val)
    ci = ci_raw.subs(xi, omega[k]).subs(k, k - j_val)
    results[j_val] = (cv, ci)

# j_val=0: subtract all oscillating terms from the full expression
# reconstructing the oscillating part in original (unshifted) k
V_osc_original = Add(*[Vn1_expanded.coeff(E(j)) * E(j) for j in js_nonzero])
I_osc_original = Add(*[In1_expanded.coeff(E(j)) * E(j) for j in js_nonzero])

cv0 = expand(Vn1_expanded - V_osc_original)   # pure DC, no E(j) left
ci0 = expand(In1_expanded - I_osc_original)

# xi shouldn't appear in DC terms, but substitute just in case
results[0] = (
    cv0.subs(xi, omega[k]),
    ci0.subs(xi, omega[k])
)

js = list(range(-2*M, 2*M+1))
V_full = Add(*[results[j_val][0] for j_val in js])
I_full = Add(*[results[j_val][1] for j_val in js])

display(Eq(V[k, n+1], V_full))
display(Eq(Ic[k, n+1], I_full))

Eq(V[k, n + 1], -Zs0*I[k, n] - Zs1(omega[k + 1])*exp(-I*theta)*I[k + 1, n] - Zs1(omega[k - 1])*exp(I*theta)*I[k - 1, n] + V[k, n])

Eq(I[k, n + 1], Yg0*Zs0*I[k, n] + Yg0*Zs1(omega[k + 1])*exp(-I*theta)*I[k + 1, n] + Yg0*Zs1(omega[k - 1])*exp(I*theta)*I[k - 1, n] - Yg0*V[k, n] + Zs0*Yg1(omega[k + 1])*exp(-I*theta)*I[k + 1, n] + Zs0*Yg1(omega[k - 1])*exp(I*theta)*I[k - 1, n] - Yg1(omega[k + 1])*exp(-I*theta)*V[k + 1, n] + Yg1(omega[k + 2])*Zs1(omega[k + 2])*exp(-2*I*theta)*I[k + 2, n] - Yg1(omega[k - 1])*exp(I*theta)*V[k - 1, n] + Yg1(omega[k - 2])*Zs1(omega[k - 2])*exp(2*I*theta)*I[k - 2, n] + 2*Yg1(omega[k])*Zs1(omega[k])*I[k, n] + I[k, n])

In [7]:
# All state symbols that can appear: V[k+j, n] and Ic[k+j, n] for j in -M..M
state_syms = [V[k+j, n]  for j in range(-M, M+1)] + \
             [Ic[k+j, n] for j in range(-M, M+1)]

V_full_collected = collect(expand(V_full), state_syms)
I_full_collected = collect(expand(I_full), state_syms)

display(Eq(V[k, n+1],  V_full_collected))
display(Eq(Ic[k, n+1], I_full_collected))

Eq(V[k, n + 1], -Zs0*I[k, n] - Zs1(omega[k + 1])*exp(-I*theta)*I[k + 1, n] - Zs1(omega[k - 1])*exp(I*theta)*I[k - 1, n] + V[k, n])

Eq(I[k, n + 1], -Yg0*V[k, n] + (Yg0*Zs1(omega[k + 1])*exp(-I*theta) + Zs0*Yg1(omega[k + 1])*exp(-I*theta))*I[k + 1, n] + (Yg0*Zs1(omega[k - 1])*exp(I*theta) + Zs0*Yg1(omega[k - 1])*exp(I*theta))*I[k - 1, n] + (Yg0*Zs0 + 2*Yg1(omega[k])*Zs1(omega[k]) + 1)*I[k, n] - Yg1(omega[k + 1])*exp(-I*theta)*V[k + 1, n] + Yg1(omega[k + 2])*Zs1(omega[k + 2])*exp(-2*I*theta)*I[k + 2, n] - Yg1(omega[k - 1])*exp(I*theta)*V[k - 1, n] + Yg1(omega[k - 2])*Zs1(omega[k - 2])*exp(2*I*theta)*I[k - 2, n])